# Multi-Agent Session Persistence

Multi-agent systems (Graph/Swarm) use session management to persist their orchestration state.

> **Caution:** Agents inside a multi-agent system must **not** have their own session manager —
> only the orchestrator should have one. The orchestrator snapshots and restores each agent
> node's state on every execution, so an agent-level session manager would conflict.
>
> **What is persisted:** orchestrator state, node execution history, shared context, and handoff
> state. Individual agent conversation histories are **not** persisted.
>
> See the [official docs](https://strandsagents.com/docs/user-guide/concepts/agents/session-management/#multi-agent-sessions).

This notebook demonstrates:
1. **Swarm** — with `S3SessionManager` on the orchestrator
2. **Graph** — with `FileSessionManager` on the orchestrator

### Prerequisites
* Complete [01-single-agent-persistence.ipynb](./01-single-agent-persistence.ipynb) — it introduces `SessionManager` and the file and S3 backends used here
* IAM permissions for Amazon S3 — this notebook creates and then deletes a bucket

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
import os
import shutil
import boto3

REGION = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
STORAGE_DIR = "./sessions"

shutil.rmtree(STORAGE_DIR, ignore_errors=True)
print(f"Region: {REGION}, Account: {ACCOUNT_ID}")
print("Session directory cleared.")

## Part 1 — Swarm with S3SessionManager

A Swarm lets agents hand off tasks to each other dynamically. The session manager
is passed to the orchestrator only — agents have no session manager of their own.

In [ ]:
S3_BUCKET = f"strands-sessions-multiagent-{ACCOUNT_ID}"
S3_PREFIX = "swarm-sessions"
SWARM_SESSION_ID = "swarm-session-demo"

s3 = boto3.client("s3", region_name=REGION)

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Bucket '{S3_BUCKET}' created.")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{S3_BUCKET}' already exists.")

In [ ]:
from strands import Agent
from strands.multiagent.swarm import Swarm
from strands.session.s3_session_manager import S3SessionManager

# Agents have no session manager — only the orchestrator does
researcher = Agent(
    system_prompt="You are a research assistant. Find and summarize information on topics.",
    name="researcher",
)

writer = Agent(
    system_prompt="You are a technical writer. Take research summaries and produce clear documentation.",
    name="writer",
)

swarm = Swarm(
    nodes=[researcher, writer],
    entry_point=researcher,
    session_manager=S3SessionManager(
        session_id=SWARM_SESSION_ID,
        bucket=S3_BUCKET,
        prefix=S3_PREFIX,
        boto_session=boto3.Session(region_name=REGION),
    ),
)

result = swarm(
    "Research the key benefits of event-driven architecture for microservices."
)
print(result)

In [ ]:
# Inspect objects saved in S3
response_s3 = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=f"{S3_PREFIX}/session_{SWARM_SESSION_ID}/",
)
print("Objects in S3:")
for obj in response_s3.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

In [ ]:
# Cleanup S3
response_s3 = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"{S3_PREFIX}/")
for obj in response_s3.get("Contents", []):
    s3.delete_object(Bucket=S3_BUCKET, Key=obj["Key"])
s3.delete_bucket(Bucket=S3_BUCKET)
print(f"Bucket '{S3_BUCKET}' deleted.")

## Part 2 — Graph with FileSessionManager

A Graph defines a deterministic pipeline where agents execute as ordered nodes.
Same pattern — session manager on the `GraphBuilder`, not on individual agents.

In [ ]:
from strands import Agent
from strands.multiagent.graph import GraphBuilder
from strands.session.file_session_manager import FileSessionManager

GRAPH_SESSION_ID = "graph-session-demo"

# Agents have no session manager — only the orchestrator does
planner = Agent(
    system_prompt="You are a planner. Respond with exactly 2 bullet points — no more.",
    name="planner",
)

executor = Agent(
    system_prompt="You are an executor. Take the plan and respond with one sentence per bullet point.",
    name="executor",
)

reviewer = Agent(
    system_prompt="You are a reviewer. Respond with one sentence: approved or needs changes and why.",
    name="reviewer",
)

# Build the graph: planner → executor → reviewer
builder = GraphBuilder()
planner_node = builder.add_node(planner)
executor_node = builder.add_node(executor)
reviewer_node = builder.add_node(reviewer)
builder.add_edge(planner_node, executor_node)
builder.add_edge(executor_node, reviewer_node)
builder.set_session_manager(
    FileSessionManager(session_id=GRAPH_SESSION_ID, storage_dir=STORAGE_DIR)
)
graph = builder.build()

result = graph("Plan adding a health check endpoint to a REST API.")
print(result)

In [ ]:
# Inspect the persisted graph session structure
graph_session_dir = os.path.join(STORAGE_DIR, f"session_{GRAPH_SESSION_ID}")
for root, dirs, files in os.walk(graph_session_dir):
    level = root.replace(graph_session_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

## Cleanup

In [ ]:
shutil.rmtree(STORAGE_DIR, ignore_errors=True)
print("Local session files cleaned up.")